# Companion notebook · Bayesian inference on the 2D Ising model

This notebook is the interactive counterpart of the laboratory guide in the README. It runs the same pipeline that `python main.py` runs, but keeps intermediate objects (the equilibrium lattices, the simulated `M(T)` data, the MCMC trace) in memory so you can inspect them cell by cell.

**How to use:**

- install `requirements.txt` from the project root,
- launch the notebook from the project root so relative paths work:

```bash
cd 01-ising-bayesian
jupyter notebook notebooks/explore.ipynb
```

- run cells top to bottom.

**Heads-up.** Running Section 3 overwrites `data/magnetization.csv` and Section 4 overwrites `data/trace.nc`. If you want to keep the canonical artifacts untouched, change the `out_csv` / `trace_out` paths in those cells before running them.

## 1 · Setup

In [ ]:
import os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
print('working directory:', os.getcwd())

import numpy as np
import matplotlib.pyplot as plt

from src import metropolis, bayesian, plots

ONSAGER_TC = 2.0 / np.log(1.0 + np.sqrt(2.0))
EXACT_BETA = 0.125
print(f'Onsager exact: Tc = {ONSAGER_TC:.4f},  β = {EXACT_BETA}')

## 2 · A picture of the two phases

Before collecting any curve, let us look at what the system actually *is* in each regime. We equilibrate two independent lattices — one well below the expected transition temperature and one well above — and print them as images.

A phase transition is an abstract concept in thermodynamics, but this picture is as concrete as it gets: on one side of `Tc` you see a dominant color, on the other side you see noise. Nothing else needs to be measured to know that *something* qualitative happens in between.

In [ ]:
def run_to_equilibrium(T, size=32, n_sweeps=2000, seed=0):
    np.random.seed(seed)
    lat = np.where(np.random.random((size, size)) < 0.5, -1, 1).astype(np.int8)
    for _ in range(n_sweeps):
        metropolis._sweep(lat, T, 1.0)
    return lat

lat_cold = run_to_equilibrium(T=1.8, seed=0)
lat_hot  = run_to_equilibrium(T=3.2, seed=1)

fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.6))
titles = [
    fr'$T = 1.8$  $(T < T_c)$ — ordered, $\langle |M| \rangle \approx {np.abs(lat_cold.mean()):.2f}$',
    fr'$T = 3.2$  $(T > T_c)$ — disordered, $\langle |M| \rangle \approx {np.abs(lat_hot.mean()):.2f}$',
]
for ax, lat, title in zip(axes, [lat_cold, lat_hot], titles):
    ax.imshow(lat, cmap='PuOr', vmin=-1, vmax=1, interpolation='nearest')
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('One equilibrium configuration in each phase  ·  32×32 lattice', fontweight='bold')
fig.tight_layout()
plt.show()

## 3 · The temperature sweep

This is the expensive step. We scan 25 temperatures between `T = 1.5` and `T = 3.5` with the same settings `main.py` uses, so the output is directly comparable to the cached `data/magnetization.csv`. Runtime on a modern laptop with Numba: ≈30–90 seconds.

After the sweep we plot the raw curve without any fit. The goal is to *see* where the transition happens before we try to extract numbers from it.

In [ ]:
results = metropolis.run_full_simulation(
    out_csv='data/magnetization.csv',
    size=32,
    n_temps=25,
    t_min=1.5, t_max=3.5,
    n_therm=2000,
    n_measure=3000,
    seed=0,
)

T  = np.array([r['T']      for r in results])
M  = np.array([r['M_mean'] for r in results])
Sd = np.array([r['M_std']  for r in results])

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.axvspan(T.min(), ONSAGER_TC, color='#7c5cff', alpha=0.06)
ax.axvspan(ONSAGER_TC, T.max(), color='#f59e0b', alpha=0.06)
ax.errorbar(T, M, yerr=Sd, fmt='o', color='#7c5cff',
            markeredgecolor='white', ecolor='#6b7280', capsize=2.5,
            label=r'$\langle |M| \rangle \pm \sigma_M$')
ax.axvline(ONSAGER_TC, color='#f472b6', ls='--', lw=1.8, label='Onsager $T_c$')
ax.set_xlabel(r'Temperature  $T$   ($J/k_B$)')
ax.set_ylabel(r'Magnetization per site  $\langle |M| \rangle$')
ax.set_title('Simulated M(T) — the "data" for the Bayesian stage', loc='left')
ax.grid(alpha=0.3)
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 4 · Bayesian inference

Default backend is the NumPy random-walk Metropolis-Hastings sampler in `src/bayesian.py` — no C compiler required. Swap for `backend='pymc'` if you have PyMC with a working PyTensor backend.

In [ ]:
trace = bayesian.run_full_inference(
    csv_in='data/magnetization.csv',
    trace_out='data/trace.nc',
    backend='numpy',
    draws=5000, tune=1000, chains=4,
    seed=0,
)

In [ ]:
import arviz as az
summary = az.summary(trace, var_names=['Tc', 'beta', 'sigma'], hdi_prob=0.95)
summary

## 5 · Convergence diagnostics

If `r_hat > 1.05` or `ess_bulk < 200` for any parameter, the posterior is untrustworthy — rerun with more draws, wider priors, or switch to NUTS. With the defaults above, all three should pass.

In [ ]:
rhat = {v: float(az.rhat(trace)[v]) for v in ['Tc', 'beta', 'sigma']}
ess  = {v: float(az.ess(trace)[v])  for v in ['Tc', 'beta', 'sigma']}
print('R_hat   :', {k: round(v, 3) for k, v in rhat.items()})
print('ESS_bulk:', {k: int(v)      for k, v in ess.items()})

## 6 · Regenerate the README figures

Calls `src.plots.plot_all`, which is exactly what runs at the end of `python main.py`. Writes the three PNGs into `figures/`.

In [ ]:
plots.plot_all(
    csv_path='data/magnetization.csv',
    trace_path='data/trace.nc',
    out_dir='figures',
    lattice_size=32,
)

## 7 · Ground-truth check

Validation rules derived from Onsager (1944):

- the 95% HDI of `β` **should contain** `1/8 = 0.125` — the exponent is universal, independent of lattice size,
- the 95% HDI of `Tc` is **expected to sit slightly above** Onsager's `2.2692` — this is the finite-size shift, well documented in Ferrenberg & Landau (1991).

If either of those expectations is violated, the sampler or the likelihood is suspect — not the physics.

In [ ]:
beta_samples = trace.posterior['beta'].values.ravel()
tc_samples   = trace.posterior['Tc'].values.ravel()

beta_hdi = az.hdi(beta_samples, hdi_prob=0.95)
tc_hdi   = az.hdi(tc_samples,   hdi_prob=0.95)

print(f'β : 95% HDI = [{beta_hdi[0]:.3f}, {beta_hdi[1]:.3f}]   '
      f'contains {EXACT_BETA:.3f}? {beta_hdi[0] <= EXACT_BETA <= beta_hdi[1]}')
print(f'Tc: 95% HDI = [{tc_hdi[0]:.3f}, {tc_hdi[1]:.3f}]   '
      f'contains {ONSAGER_TC:.4f}? {tc_hdi[0] <= ONSAGER_TC <= tc_hdi[1]}   '
      f'(an upward shift on a finite 32×32 lattice is expected)')